# Milestone 1 Exploration and Retrieval Corpus Preparation

This notebook documents the **data exploration and preprocessing** work for Milestone 1 of the Amazon product query assistant project.

For this milestone, we use a **representative sample** from the **Video_Games** category of the Amazon Reviews 2023 dataset.  
In this version of the workflow, the sample corpus is used not only for exploration, but also for:

- **Step 2: BM25 search**
- **Step 3: Semantic search**
- saving the retrieval artifacts locally

This notebook is designed to stay manageable on a local machine while still following the milestone requirements.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.utils import load_jsonl, build_retrieval_text, pick_join_key, save_corpus
from src.bm25 import BM25Retriever
from src.semantic import SemanticRetriever

## Load raw files

The project expects these files in `data/raw/`:

- `Video_Games.jsonl`
- `meta_Video_Games.jsonl`

The review file contains user reviews, ratings, and review titles.  
The metadata file contains product-level information such as title, description, features, categories, and price.


In [2]:
reviews_path = Path("../data/raw/Video_Games.jsonl")
meta_path = Path("../data/raw/meta_Video_Games.jsonl")

reviews_df = load_jsonl(reviews_path)
meta_df = load_jsonl(meta_path)

print("Reviews shape:", reviews_df.shape)
print("Metadata shape:", meta_df.shape)

Reviews shape: (4624615, 10)
Metadata shape: (137269, 16)


## Sample review records

In [3]:
reviews_df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4.0,It’s pretty sexual. Not my fav,I’m playing on ps5 and it’s interesting. It’s...,[],B07DJWBYKP,B07DK1H3H5,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1608186804795,0,True
1,5.0,Good. A bit slow,Nostalgic fun. A bit slow. I hope they don’t...,[],B00ZS80PC2,B07SRWRH5D,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1587051114941,1,False
2,5.0,... an order for my kids & they have really en...,This was an order for my kids & they have real...,[],B01FEHJYUU,B07MFMFW34,AGXVBIUFLFGMVLATYXHJYL4A5Q7Q,1490877431000,0,True
3,5.0,Great alt to pro controller,"These work great, They use batteries which is ...",[],B07GXJHRVK,B0BCHWZX95,AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q,1577637634017,0,True
4,5.0,solid product,I would recommend to anyone looking to add jus...,[],B00HUWA45W,B00HUWA45W,AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q,1427591932000,0,True


## Sample metadata records

In [4]:
meta_df.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Video Games,Dash 8-300 Professional Add-On,5.0,1,[Features Dash 8-300 and 8-Q300 ('Q' rollout l...,[The Dash 8-300 Professional Add-On lets you p...,None,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aerosoft,"[Video Games, PC, Games]",{'Pricing': 'The strikethrough price is the Li...,B000FH0MHO,None,NaN,NaN
1,Video Games,Phantasmagoria: A Puzzle of Flesh,4.1,18,[Windows 95],[],None,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sierra,"[Video Games, PC, Games]","{'Best Sellers Rank': {'Video Games': 137612, ...",B00069EVOG,None,NaN,NaN
2,Video Games,NBA 2K17 - Early Tip Off Edition - PlayStation 4,4.3,223,[The #1 rated NBA video game simulation series...,[Following the record-breaking launch of NBA 2...,58.0,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'NBA 2K17 - Kobe: Haters vs Players...,2K,"[Video Games, PlayStation 4, Games]","{'Release date': 'September 16, 2016', 'Best S...",B00Z9TLVK0,None,NaN,NaN
3,Video Games,Nintendo Selects: The Legend of Zelda Ocarina ...,4.9,22,[Authentic Nintendo Selects: The Legend of Zel...,[],37.42,[{'thumb': 'https://m.media-amazon.com/images/...,[],Amazon Renewed,"[Video Games, Legacy Systems, Nintendo Systems...","{'Best Sellers Rank': {'Video Games': 51019, '...",B07SZJZV88,None,NaN,NaN
4,Video Games,Thrustmaster Elite Fitness Pack for Nintendo Wii,3.0,3,"[Includes (9) Total Accessories, Pedometer, Wi...",[The Thrustmaster Motion Plus Elite Fitness Pa...,None,[{'thumb': 'https://m.media-amazon.com/images/...,[],THRUSTMASTER,"[Video Games, Legacy Systems, Nintendo Systems...","{'Release date': 'November 1, 2009', 'Pricing'...",B002WH4ZJG,None,NaN,NaN


## Column overview

In [5]:
print("Review columns:")
print(reviews_df.columns.tolist())

print("\nMetadata columns:")
print(meta_df.columns.tolist())

Review columns:
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Metadata columns:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']


## Missingness in key fields

In [6]:
key_cols_reviews = [col for col in ["parent_asin", "asin", "title", "text", "rating"] if col in reviews_df.columns]
key_cols_meta = [col for col in ["parent_asin", "asin", "title", "description", "features", "categories", "price"] if col in meta_df.columns]

print("Missing values in review fields:")
print(reviews_df[key_cols_reviews].isna().sum())

print("\nMissing values in metadata fields:")
print(meta_df[key_cols_meta].isna().sum())

Missing values in review fields:
parent_asin    0
asin           0
title          0
text           0
rating         0
dtype: int64

Missing values in metadata fields:
parent_asin        0
title              0
description        0
features           0
categories         0
price          75261
dtype: int64


## Selection and justification of fields for retrieval

For this project, we build **one retrieval document per review**, enriched with product metadata.

### Selected fields from reviews
We use these review-level fields:

- `parent_asin` or `asin`: used as the join key between reviews and product metadata
- `title` → renamed to `review_title`: gives a short summary of the user’s opinion
- `text`: the main review content and the most important source of user intent signals
- `rating`: not used directly for matching, but kept as display metadata for later retrieval results and app output

### Selected fields from metadata
We use these product-level fields:

- `title` → renamed to `product_title`: important for direct product matching and for identifying the item clearly
- `description`: adds broader product context that may not appear in reviews
- `features`: useful for keyword matching because it often lists product capabilities or components
- `categories`: helps connect queries to product type and genre information
- `price`: not used for retrieval in Milestone 1, but kept as metadata for later use in display or filtering

### Why this field combination makes sense
This field selection is a good fit for Milestone 1 for several reasons:

1. **The app needs review-level output.**  
   The milestone app is expected to show review text and rating, so using one row per review makes the retrieval output easy to display later.

2. **Both retrievers should use the same document unit.**  
   BM25 and semantic retrieval are easier to compare fairly when both methods search over the same review-level corpus.

3. **Metadata improves matching quality.**  
   Some user queries may mention product names, genres, or features that do not appear strongly in the review text alone. Adding product title, categories, features, and description gives the retrievers more useful signals.

4. **The corpus contains both user language and product language.**  
   Reviews reflect how people describe gameplay and experience, while metadata reflects how products are formally described. Combining both makes retrieval more robust.

### Final retrieval text design
The final `retrieval_text` field is built from:

- product title
- categories
- features
- description
- review title
- review text

This gives the retrievers both product-level and review-level context in one searchable text field.


## Preprocessing decisions

For Milestone 1, we use simple and reproducible preprocessing rather than aggressive NLP cleaning.

### Main preprocessing choices

- lowercase all text
- remove most punctuation
- normalize repeated whitespace
- use whitespace tokenization for BM25
- keep the semantic retriever on the same cleaned retrieval text for consistency

### Why these choices are reasonable

**1. Lowercasing**  
Lowercasing reduces unnecessary variation between tokens such as `Game`, `game`, and `GAME`. This is especially helpful for BM25 because it avoids splitting the same concept into multiple token forms.

**2. Removing most punctuation**  
Punctuation usually does not help retrieval in this project. Removing it simplifies the text and improves token consistency for BM25.

**3. Normalizing whitespace**  
This makes the corpus cleaner and avoids issues caused by repeated spaces, line breaks, or formatting artifacts.

**4. Simple whitespace tokenization for BM25**  
The milestone instructions only require a simple tokenization function at minimum. Using whitespace tokenization keeps the system easy to understand and reproducible.

**5. Shared document content across retrievers**  
BM25 and semantic retrieval should search over the same underlying document unit and roughly the same text content. This makes later comparison more meaningful.

### What we are not doing yet

For Milestone 1, we are not adding more advanced preprocessing such as:

- stemming
- lemmatization
- heavy stopword filtering
- query rewriting
- reranking
- LLM-based cleaning or summarization

This keeps the pipeline simple while still satisfying the retrieval requirements of the milestone.


## Build the retrieval corpus step by step

In [7]:
join_key = pick_join_key(reviews_df, meta_df)
print("Join key:", join_key)

Join key: parent_asin


In [8]:
reviews = reviews_df.copy()
meta = meta_df.copy()

if "title" in reviews.columns:
    reviews = reviews.rename(columns={"title": "review_title"})
if "title" in meta.columns:
    meta = meta.rename(columns={"title": "product_title"})

print("Reviews columns ready.")
print("Meta columns ready.")

Reviews columns ready.
Meta columns ready.


In [9]:
meta_cols = [
    col for col in [join_key, "product_title", "description", "features", "categories", "price"]
    if col in meta.columns
]

meta_small = meta[meta_cols].drop_duplicates(subset=[join_key]).copy()

print("Reduced metadata shape:", meta_small.shape)
meta_small.head()

Reduced metadata shape: (137269, 6)


,parent_asin,product_title,description,features,categories,price
0,B000FH0MHO,Dash 8-300 Professional Add-On,[The Dash 8-300 Professional Add-On lets you p...,[Features Dash 8-300 and 8-Q300 ('Q' rollout l...,"[Video Games, PC, Games]",None
1,B00069EVOG,Phantasmagoria: A Puzzle of Flesh,[],[Windows 95],"[Video Games, PC, Games]",None
2,B00Z9TLVK0,NBA 2K17 - Early Tip Off Edition - PlayStation 4,[Following the record-breaking launch of NBA 2...,[The #1 rated NBA video game simulation series...,"[Video Games, PlayStation 4, Games]",58.0
3,B07SZJZV88,Nintendo Selects: The Legend of Zelda Ocarina ...,[],[Authentic Nintendo Selects: The Legend of Zel...,"[Video Games, Legacy Systems, Nintendo Systems...",37.42
4,B002WH4ZJG,Thrustmaster Elite Fitness Pack for Nintendo Wii,[The Thrustmaster Motion Plus Elite Fitness Pa...,"[Includes (9) Total Accessories, Pedometer, Wi...","[Video Games, Legacy Systems, Nintendo Systems...",None


In [10]:
review_cols = [
    col for col in [join_key, "review_title", "text", "rating"]
    if col in reviews.columns
]

reviews_small = reviews[review_cols].copy()

print("Reduced reviews shape:", reviews_small.shape)
reviews_small.head()

Reduced reviews shape: (4624615, 4)


,parent_asin,review_title,text,rating
0,B07DK1H3H5,It’s pretty sexual. Not my fav,I’m playing on ps5 and it’s interesting. It’s...,4.0
1,B07SRWRH5D,Good. A bit slow,Nostalgic fun. A bit slow. I hope they don’t...,5.0
2,B07MFMFW34,... an order for my kids & they have really en...,This was an order for my kids & they have real...,5.0
3,B0BCHWZX95,Great alt to pro controller,"These work great, They use batteries which is ...",5.0
4,B00HUWA45W,solid product,I would recommend to anyone looking to add jus...,5.0


In [11]:
reviews_small["text"] = reviews_small["text"].fillna("")
reviews_small = reviews_small[reviews_small["text"].str.len() >= 20].copy()

if "review_title" not in reviews_small.columns:
    reviews_small["review_title"] = ""

print("Filtered reviews shape:", reviews_small.shape)

Filtered reviews shape: (4078697, 4)


In [12]:
merged = reviews_small.merge(meta_small, on=join_key, how="left")

if "product_title" not in merged.columns:
    merged["product_title"] = ""

print("Merged shape:", merged.shape)
merged.head()

Merged shape: (4078697, 9)


,parent_asin,review_title,text,rating,product_title,description,features,categories,price
0,B07DK1H3H5,It’s pretty sexual. Not my fav,I’m playing on ps5 and it’s interesting. It’s...,4.0,Cyberpunk 2077 - PC [Game Download Code in Box],"[Cyberpunk 2077 is an open world, an action ad...",[Become a cyberpunk an urban mercenary equippe...,"[Video Games, PC, Games]",None
1,B07SRWRH5D,Good. A bit slow,Nostalgic fun. A bit slow. I hope they don’t...,5.0,Final Fantasy VII: Remake - PlayStation 4,[A spectacular reimagining of one of the most ...,[Explore a dark & eclectic world – Dive deep i...,"[Video Games, PlayStation 4, Games]",25.95
2,B07MFMFW34,... an order for my kids & they have really en...,This was an order for my kids & they have real...,5.0,Sid Meier’s Civilization VI: Rise and Fall [On...,[Civilization VI is a game about building an e...,[Civilization VI: Rise and Fall introduces eig...,"[Video Games, PC, Games]",29.99
3,B0BCHWZX95,Great alt to pro controller,"These work great, They use batteries which is ...",5.0,PowerA Enhanced Wireless Controller for Ninten...,[Play your favorite Nintendo Switch games like...,"[Bluetooth wireless freedom, Ergonomic design ...","[Video Games, Nintendo Switch, Accessories, Co...",67.61
4,B00HUWA45W,solid product,I would recommend to anyone looking to add jus...,5.0,KontrolFreek FPS Freek CQC Signature - Xbox One,[],"[Better control = Higher accuracy in FPSs, Mor...","[Video Games, Xbox One, Accessories]",None


## Create a representative sample corpus

The full dataset is large, so this notebook uses a representative sample for the rest of the project.

You can change `SAMPLE_SIZE` if needed, but a moderate size is recommended so both BM25 and semantic search remain practical on a local machine.


In [13]:
SAMPLE_SIZE = 50000

corpus_sample = merged.iloc[:SAMPLE_SIZE].copy()

print("Corpus sample shape:", corpus_sample.shape)

Corpus sample shape: (50000, 9)


In [14]:
corpus_sample["retrieval_text"] = corpus_sample.apply(build_retrieval_text, axis=1)
corpus_sample = corpus_sample[corpus_sample["retrieval_text"].str.len() > 0].copy()
corpus_sample = corpus_sample.reset_index(drop=True)
corpus_sample["doc_id"] = [f"doc_{i}" for i in range(len(corpus_sample))]

print("Corpus sample after retrieval text:", corpus_sample.shape)

Corpus sample after retrieval text: (50000, 11)


In [15]:
keep_cols = [
    col for col in [
        "doc_id",
        join_key,
        "product_title",
        "review_title",
        "text",
        "rating",
        "price",
        "retrieval_text",
        "description",
        "features",
        "categories",
    ]
    if col in corpus_sample.columns
]

corpus_sample = corpus_sample[keep_cols].copy()
corpus_sample.head()

,doc_id,parent_asin,product_title,review_title,text,rating,price,retrieval_text,description,features,categories
0,doc_0,B07DK1H3H5,Cyberpunk 2077 - PC [Game Download Code in Box],It’s pretty sexual. Not my fav,I’m playing on ps5 and it’s interesting. It’s...,4.0,None,cyberpunk 2077 pc game download code in box vi...,"[Cyberpunk 2077 is an open world, an action ad...",[Become a cyberpunk an urban mercenary equippe...,"[Video Games, PC, Games]"
1,doc_1,B07SRWRH5D,Final Fantasy VII: Remake - PlayStation 4,Good. A bit slow,Nostalgic fun. A bit slow. I hope they don’t...,5.0,25.95,final fantasy vii remake playstation 4 video g...,[A spectacular reimagining of one of the most ...,[Explore a dark & eclectic world – Dive deep i...,"[Video Games, PlayStation 4, Games]"
2,doc_2,B07MFMFW34,Sid Meier’s Civilization VI: Rise and Fall [On...,... an order for my kids & they have really en...,This was an order for my kids & they have real...,5.0,29.99,sid meier s civilization vi rise and fall onli...,[Civilization VI is a game about building an e...,[Civilization VI: Rise and Fall introduces eig...,"[Video Games, PC, Games]"
3,doc_3,B0BCHWZX95,PowerA Enhanced Wireless Controller for Ninten...,Great alt to pro controller,"These work great, They use batteries which is ...",5.0,67.61,powera enhanced wireless controller for ninten...,[Play your favorite Nintendo Switch games like...,"[Bluetooth wireless freedom, Ergonomic design ...","[Video Games, Nintendo Switch, Accessories, Co..."
4,doc_4,B00HUWA45W,KontrolFreek FPS Freek CQC Signature - Xbox One,solid product,I would recommend to anyone looking to add jus...,5.0,None,kontrolfreek fps freek cqc signature xbox one ...,[],"[Better control = Higher accuracy in FPSs, Mor...","[Video Games, Xbox One, Accessories]"


In [16]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

save_corpus(corpus_sample, processed_dir / "video_games_corpus_sample.parquet")
print("Saved sample corpus to ../data/processed/video_games_corpus_sample.parquet")

Saved sample corpus to ../data/processed/video_games_corpus_sample.parquet


## Step 2: BM25 Search on the representative sample

This section builds the BM25 retriever on the sample corpus and saves the required BM25 artifacts locally.


In [17]:
bm25 = BM25Retriever(corpus_sample)
bm25.save(
    processed_dir / "bm25_sample_index.pkl",
    processed_dir / "bm25_sample_tokens.pkl",
)

print("Saved BM25 sample artifacts:")
print("- ../data/processed/bm25_sample_index.pkl")
print("- ../data/processed/bm25_sample_tokens.pkl")

Saved BM25 sample artifacts:
- ../data/processed/bm25_sample_index.pkl
- ../data/processed/bm25_sample_tokens.pkl


In [18]:
bm25_query = "best racing game with fun tracks"
bm25_results = bm25.search(bm25_query, top_k=5)

print("BM25 query:", bm25_query)
bm25_results[["product_title", "review_title", "rating", "bm25_score"]].head()

BM25 query: best racing game with fun tracks


,product_title,review_title,rating,bm25_score
0,LittleBigPlanet Karting - Playstation 3,Fun game for everyone,5.0,24.851332
1,LittleBigPlanet Karting - Playstation 3,Seller came thru!,5.0,24.825243
2,LittleBigPlanet Karting - Playstation 3,Fun!,5.0,24.802589
3,LittleBigPlanet Karting - Playstation 3,A present for my adult daughter on her birthday,5.0,24.638145
4,Champion Jockey: G1 Jockey and Gallop Racer - ...,Two Stars,2.0,24.388155


## Step 3: Semantic Search on the representative sample

This section builds the semantic retriever on the sample corpus using sentence embeddings and FAISS, then saves the semantic artifacts locally.

Note: this step may take a while the first time because the embedding model may need to download.


In [ ]:
semantic = SemanticRetriever(corpus_sample)
semantic.build_index()
semantic.save(
    processed_dir / "faiss_sample.index",
    processed_dir / "semantic_sample_metadata.pkl",
)

print("Saved semantic sample artifacts:")
print("- ../data/processed/faiss_sample.index")
print("- ../data/processed/semantic_sample_metadata.pkl")

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
semantic_query = "story-rich scary game with dark atmosphere"
semantic_results = semantic.search(semantic_query, top_k=5)

print("Semantic query:", semantic_query)
semantic_results[["product_title", "review_title", "rating", "semantic_score"]].head()

## Save a few sample retrieval outputs

These CSV files make it easy to inspect and share example outputs from both retrieval methods.


In [ ]:
bm25_results.to_csv(processed_dir / "bm25_sample_results.csv", index=False)
semantic_results.to_csv(processed_dir / "semantic_sample_results.csv", index=False)

print("Saved:")
print("- ../data/processed/bm25_sample_results.csv")
print("- ../data/processed/semantic_sample_results.csv")

## Summary

This notebook now covers:

- Step 1: data exploration and preprocessing
- Step 2: BM25 search on the representative sample
- Step 3: semantic search on the representative sample

Saved sample outputs include:

- `video_games_corpus_sample.parquet`
- `bm25_sample_index.pkl`
- `bm25_sample_tokens.pkl`
- `faiss_sample.index`
- `semantic_sample_metadata.pkl`
- `bm25_sample_results.csv`
- `semantic_sample_results.csv`

This sample-based workflow is suitable if you plan to use the representative sample throughout the rest of the project.
